# 10장 지식을 구조화하는 다른 방법

이 장의 예제와 실습을 실행하는 노트북입니다. 개념 설명은 HTML 교재 `10_지식을_구조화하는_다른_방법.html`에 있습니다. 위에서부터 차례로 실행합니다. 실습의 풀이는 교재의 실습 상자에 접혀 있습니다.

## 준비

이 노트북은 `노트북` 폴더에서 실행합니다. 같은 폴더의 `llm.py`가 API 호출을 맡습니다. Python 3.10 이상이 필요하고, 패키지는 다음 명령으로 설치합니다.

```bash
python -m pip install anthropic openai pandas pydantic openpyxl jupyter
```

장마다 더 필요한 패키지가 있으면 아래 "준비 (2)" 셀 안의 `# 설치:` 주석에 명령이 있습니다. 그 명령도 터미널에서 실행하고 커널을 다시 시작합니다.

### API 키는 코드에 적지 않습니다

API 키는 API를 호출한 사람을 식별하는 문자열입니다. 사용 기록과 요금은 키별로 집계됩니다. 키를 가진 사람은 누구나 그 키로 호출할 수 있고, 요금은 키의 주인에게 청구됩니다.

코드에 적은 키는 파일을 공유하거나 저장소에 올리는 순간 함께 퍼집니다. 그래서 키는 환경 변수에 두고 코드에서는 환경 변수를 읽습니다.

> **주의**
>
> 키를 메신저, 화면 공유, 캡처 이미지에 노출하지 않습니다. 노출됐다면 바로 폐기하고 새로 발급받습니다.

### 환경 변수 설정

`llm.py`는 아래 환경 변수를 읽어 어느 API로 보낼지 고릅니다.

**Anthropic**

| 환경 변수 | 값 |
| --- | --- |
| `ANTHROPIC_API_KEY` | 발급받은 키 |

모델 이름은 설정하지 않아도 됩니다. 소형, 기본, 대형 모델 등급에 현행 Claude 모델이 기본값으로 들어 있습니다.

**사내 API(OpenAI 호환)**

| 환경 변수 | 값 |
| --- | --- |
| `LLM_PROVIDER` | `openai` |
| `OPENAI_BASE_URL` | 사내 API 주소. 보통 `/v1`로 끝납니다 |
| `OPENAI_API_KEY` | 사내 API 키 |
| `LLM_MODEL_SMALL` | 소형 모델 이름 |
| `LLM_MODEL_DEFAULT` | 기본 모델 이름 |
| `LLM_MODEL_LARGE` | 대형 모델 이름 |
| `LLM_STRUCTURED` | 구조화 출력을 요청하는 방식. 설정하지 않으면 `json_schema`입니다. 서버가 이 방식을 지원하지 않을 때만 바꿉니다(6장 "구조화 출력") |

**PC에서 모델을 내려받을 때(공개 LLM, 임베딩 모델, 리랭커)**

| 환경 변수 | 값 |
| --- | --- |
| `HF_ENDPOINT` | 사내 Hugging Face 프록시 주소. 비워 두면 huggingface.co에서 받습니다. 이 값도 Jupyter를 시작하기 전에 설정합니다 |
| `EMBED_MODEL` | 설정하지 않아도 됩니다. 다른 임베딩 모델을 쓰거나 모델 파일이 든 폴더를 따로 받았을 때, 모델 이름이나 폴더 경로를 넣습니다 |

사내에서 쓸 수 있는 모델이 하나뿐이면 세 변수에 같은 이름을 넣습니다. 그러면 모델 등급을 비교하는 예제와 실습에서는 차이가 나지 않습니다.

환경 변수는 Jupyter를 시작하기 전에 설정합니다. `setx`로 설정한 값은 그 뒤에 새로 연 창에만 적용됩니다. Windows PowerShell에서 다음과 같이 설정한 뒤 터미널을 새로 열고, 그 터미널에서 Jupyter를 시작합니다. VS Code로 노트북을 연다면 VS Code 창을 모두 닫았다가 다시 엽니다.

Anthropic API를 쓸 때:

```powershell
setx ANTHROPIC_API_KEY "발급받은_키"
```

사내 API를 쓸 때(위 표의 나머지 변수도 같은 방법으로 설정합니다):

```powershell
setx LLM_PROVIDER openai
setx OPENAI_BASE_URL "사내_API_주소"
setx OPENAI_API_KEY "사내_API_키"
```

`setx`로 설정한 값은 모든 장의 노트북에 그대로 적용됩니다.

환경 변수를 미리 설정하지 못했다면 아래 "준비 (1)" 셀이 키, 주소, 모델 이름을 물어봅니다. 입력한 값은 화면에 보이지 않고 노트북 파일에도 저장되지 않습니다. 이 값은 커널이 살아 있는 동안만 쓰이므로, 노트북을 새로 열거나 커널을 다시 시작하면 다시 입력합니다. 단, 사내 API를 쓰려면 `LLM_PROVIDER=openai`만은 Jupyter를 시작하기 전에 설정해야 합니다. "준비 (1)" 셀이 이 값을 보고 어떤 값을 물을지 정하기 때문입니다. 이 값이 없으면 `llm.py`는 Anthropic API로 요청을 보냅니다.

### 연결 확인

아래 두 셀을 차례로 실행합니다. 두 셀이 오류 없이 끝나고 마지막 줄에 모델의 응답(보통 `준비 완료`)이 나오면 연결된 것입니다. 응답 문구는 모델에 따라 조금 다를 수 있습니다.

In [ ]:
# 준비 (1) 환경 변수가 비어 있으면 여기서 입력합니다
import os
from getpass import getpass

import llm

needed = (['ANTHROPIC_API_KEY'] if llm.provider() == 'anthropic'
          else ['OPENAI_BASE_URL', 'OPENAI_API_KEY', 'LLM_MODEL_SMALL', 'LLM_MODEL_DEFAULT',
                'LLM_MODEL_LARGE'])
for name in needed:
    if not os.environ.get(name):
        os.environ[name] = getpass(f'{name}: ').strip()
print('사용하는 API:', llm.provider())

In [ ]:
# 준비 (2) 공통으로 쓰는 것을 불러오고 연결을 확인합니다
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd

DATA = 'data'   # 재료 폴더. 장마다 쓰는 재료는 이 아래에 있습니다

# 설치: python -m pip install pypdf sentence-transformers
# 10장의 예제와 실습이 쓰는 문서 코퍼스(가상 회사 문서 120건)와 검색 함수입니다
import csv
import html
import re
from collections import Counter
from pathlib import Path

import numpy as np
from pypdf import PdfReader

import embed

CORPUS = Path(DATA) / 'corpus'
with open(f'{DATA}/corpus_목록.csv', encoding='utf-8-sig') as file:
    CATALOG = {row['id']: row for row in csv.DictReader(file)}    # 문서 번호별 메타데이터(제목, 종류, 관리 부서, 상태 등)


def read_doc(doc_id):
    """문서 하나의 본문 글을 돌려줍니다. Markdown은 front matter를 빼고, PDF와 HTML은 글만 뽑습니다(8장)"""
    path = CORPUS / CATALOG[doc_id]['file']
    if path.suffix == '.pdf':
        return '\n'.join(page.extract_text() for page in PdfReader(path).pages)
    text = path.read_text(encoding='utf-8')
    if path.suffix == '.html':
        text = re.sub(r'<(script|style|nav|footer)\b.*?</\1>', ' ', text, flags=re.S)   # 메뉴와 꼬리말을 뺍니다
        text = re.sub(r'<[^>]+>', ' ', text)
        return html.unescape(re.sub(r'[ \t]+', ' ', re.sub(r'\n\s*\n+', '\n', text)).strip())
    return re.sub(r'^---\n.*?\n---\n', '', text, flags=re.S).strip()


def make_chunks(size=500, overlap=100):
    """유효한 문서를 글자 수 기준으로 잘라 청크 목록을 만듭니다. 청크마다 문서 번호와 제목을 붙입니다"""
    chunks = []
    for doc_id, row in CATALOG.items():
        if row['status'] != '유효':                   # 폐기된 구판은 검색 대상에서 뺍니다(8장 메타데이터 필터)
            continue
        text = read_doc(doc_id)
        for start in range(0, max(len(text) - overlap, 1), size - overlap):
            chunks.append({'id': doc_id, 'text': f"{row['title']}\n{text[start:start + size]}"})
    return chunks


CHUNKS = make_chunks()
VECTORS = embed.passages([chunk['text'] for chunk in CHUNKS])     # 처음 한 번 1~2분 걸립니다


def search(question, k=5):
    """질문과 코사인 유사도가 높은 청크 k개를 돌려줍니다(7장 임베딩 검색)"""
    scores = VECTORS @ embed.query(question)
    return [dict(CHUNKS[index], score=float(scores[index])) for index in np.argsort(-scores)[:k]]


RAG_SYSTEM = '너는 후공정 라인의 사내 문서 도우미다. 주어진 문서에 근거가 있는 것만 답하고, 근거 문서 번호를 적는다.'

# 위키 편집 규칙. 코딩 에이전트로 위키를 만들 때는 이 글을 규칙 파일(14장)에 적습니다
WIKI_RULES = """너는 사내 문서로 위키를 만들고 고치는 편집자다. 다음 규칙을 지킨다.
1. 원본에 있는 내용만 쓴다. 원본에서 확인할 수 없는 내용은 쓰지 않는다.
2. 사실을 적은 문장마다 끝에 출처 문서 번호를 [QIR-2025-031]처럼 붙인다.
3. 원본끼리 값이나 기준이 다르면 한쪽으로 정하지 않는다. 그 문장 앞에 [검토 필요]를 붙이고 양쪽 값과 출처를 모두 적는다.
4. 페이지는 Markdown으로 쓰고 "## 요약", "## 경과", "## 현재 기준", "## 관련 문서" 절을 둔다.
5. 다른 위키 페이지가 될 만한 낱말은 [[페이지 이름]]으로 잇는다."""

# 품질 이슈 보고서에서 뽑을 개체와 관계(온톨로지). 사람이 먼저 정합니다
RELATIONS = ['제품', '발생 설비', '담당 부서', '개정 문서', '상태']
ONTOLOGY = """개체 종류
- 보고서: 품질 이슈 보고서. QIR-2026-014처럼 문서 번호로 적는다
- 제품: QX48, SP8, BG256처럼 제품 코드로 적는다
- 설비: W-07처럼 설비 번호만 적는다. "와이어 본더" 같은 설비 이름은 붙이지 않는다
- 부서: 조립기술팀처럼 팀 이름만 적는다
- 문서: SOP-PL-001처럼 문서 번호만 적는다
관계(주어 → 대상)
- 제품: 보고서 → 불량이 난 제품
- 발생 설비: 보고서 → 불량이 난 설비. 설비가 여럿이면 하나씩 따로 적는다
- 담당 부서: 보고서 → 담당자의 부서
- 개정 문서: 보고서 → 재발 방지를 위해 새로 만들거나 고친 문서
- 상태: 보고서 → "진행 중" 또는 "종결"
규칙: 원본에 적힌 관계만 뽑는다. 해당하는 값이 원본에 없으면 그 관계는 만들지 않는다."""
TRIPLE_SCHEMA = {
    'type': 'object', 'additionalProperties': False, 'required': ['triples'],
    'properties': {'triples': {'type': 'array', 'items': {
        'type': 'object', 'additionalProperties': False, 'required': ['주어', '관계', '대상'],
        'properties': {'주어': {'type': 'string'}, '관계': {'type': 'string', 'enum': RELATIONS},
                       '대상': {'type': 'string'}}}}}}

# 사람이 원본을 읽고 정한 정답 트리플(예제 10-3에서 추출 결과와 비교). 보고서마다 (제품, 담당 부서, 상태, 발생 설비, 개정 문서)
_ANSWERS = {
    'QIR-2025-031': ('QX48', '몰딩기술팀', '종결', 'M-02', 'SOP-MD-001'),
    'QIR-2025-044': ('BG256', '웨이퍼가공팀', '종결', 'D-03', 'MAN-DS-001 SOP-DC-001'),
    'QIR-2026-002': ('QX48', '몰딩기술팀', '종결', 'T-02', 'SOP-TF-001 MAN-TF-001'),
    'QIR-2026-004': ('SP8', '조립기술팀', '종결', 'W-05', ''),
    'QIR-2026-007': ('QX48', '몰딩기술팀', '종결', 'M-01', 'SOP-MD-002'),
    'QIR-2026-009': ('QX48', '품질보증팀', '종결', 'H-04', 'SOP-ESD-001'),
    'QIR-2026-011': ('QX48', '조립기술팀', '종결', 'B-02', 'MAN-DB-001'),
    'QIR-2026-012': ('QX48', '몰딩기술팀', '종결', 'L-02', ''),
    'QIR-2026-013': ('SP8', '조립기술팀', '종결', 'B-05', ''),
    'QIR-2026-014': ('QX48', '조립기술팀', '종결', 'W-07', 'SOP-PL-001 MAN-WB-001'),
    'QIR-2026-015': ('QX48', '테스트기술팀', '종결', 'H-07', 'SOP-PT-002 SOP-PT-001'),
    'QIR-2026-016': ('BG256', '조립기술팀', '종결', 'A-01', 'MAN-BA-001 SOP-BA-001'),
    'QIR-2026-017': ('BG256', '품질보증팀', '종결', 'DC-03', 'SOP-MS-001'),
    'QIR-2026-018': ('BG256', '몰딩기술팀', '진행 중', 'M-03', ''),
    'QIR-2026-019': ('QX48', '몰딩기술팀', '진행 중', 'M-01 M-02', ''),
    'QIR-2026-020': ('SP8', '웨이퍼가공팀', '진행 중', 'G-02', ''),
    'QIR-2026-021': ('BG256', '조립기술팀', '진행 중', 'A-01', ''),
    'QIR-2026-022': ('SP8', '품질보증팀', '진행 중', '', ''),
}
ANSWER_TRIPLES = {(report, relation, target)
                  for report, (product, dept, status, equipment, docs) in _ANSWERS.items()
                  for relation, targets in [('제품', product), ('담당 부서', dept), ('상태', status),
                                            ('발생 설비', equipment), ('개정 문서', docs)]
                  for target in ([targets] if relation in ('제품', '담당 부서', '상태') else targets.split())}


print(f'문서 {len(CATALOG)}건, 청크 {len(CHUNKS)}개')


for alias in ['소형', '기본', '대형']:
    try:
        print(f'{alias} 모델:', llm.model_name(alias))
    except RuntimeError:
        print(f'{alias} 모델: 정하지 않음')

reply = llm.send([{'role': 'user', 'content': '"준비 완료"라고만 답해 주세요.'}], model='소형', max_tokens=200)
print(reply.text)

오류가 나면 아래 표에서 원인을 찾습니다.

| 나온 문구 | 원인과 조치 |
| --- | --- |
| `환경 변수 ...의 값이 비어 있습니다` | 환경 변수를 설정하지 않았습니다. 변수 이름의 철자를 확인하고 "준비 (1)" 셀을 다시 실행합니다 |
| `환경 변수 LLM_MODEL_...에 '소형' 모델 이름을 넣어 주세요` | 사내 API를 쓰는데 모델 이름을 비워 두었습니다. "준비 (1)" 셀을 다시 실행해 모델 이름을 입력합니다 |
| `401 auth` | 키가 틀렸습니다. "준비 (1)" 셀은 값이 비어 있을 때만 묻습니다. 셀에서 입력했다면 커널을 다시 시작한 뒤 "준비 (1)" 셀에서 앞뒤 공백이나 따옴표 없이 다시 입력합니다. `setx`로 설정했다면 값을 다시 설정하고 Jupyter를 다시 시작합니다 |
| `403 permission` | 키에 그 모델이나 기능을 쓸 권한이 없습니다. 키를 발급한 곳에 권한을 확인합니다 |
| `404 not_found` | 모델 이름이나 API 주소가 틀렸습니다. 고치는 방법은 `401 auth`와 같습니다 |
| `connection` | 서버에 연결하지 못했습니다. 주소와 사내망 연결을 확인합니다 |
| `timeout` | 정해진 시간 안에 응답이 오지 않았습니다. 잠시 뒤 다시 실행합니다 |
| `ModuleNotFoundError: No module named '...'` | 패키지를 설치하지 않았습니다. "준비 (2)" 셀의 `# 설치:` 주석에 있는 명령을 터미널에서 실행하고 커널을 다시 시작합니다 |
| `NameError` | 앞 셀을 건너뛰었거나 커널을 다시 시작했습니다. 위에서부터 차례로 다시 실행합니다 |

## 예제 10-1. RAG에 집계 질문하기

교재의 절: "RAG가 답하기 어려운 질문"

진행 중인 품질 이슈 보고서의 건수를 RAG로 묻습니다. 검색된 청크 5개가 어느 문서에서 왔는지, 모델이 몇 건이라고 답하는지 보고, 2026년 보고서 16건을 모두 열어 센 정답과 비교합니다. 이 셀은 API를 1번 호출합니다.

In [ ]:
question = '2026년 품질 이슈 보고서 가운데 진행 상태가 "진행 중"인 것은 몇 건이고, 어느 보고서인가요?'
hits = search(question, k=5)                                   # 질문과 가까운 청크 5개(top-k의 k=5)
print('검색된 청크의 문서:', [hit['id'] for hit in hits])
context = '\n\n'.join(f"[{hit['id']}] {hit['text']}" for hit in hits)
reply = llm.ask(f'{context}\n\n질문: {question}', system=RAG_SYSTEM, model='소형', max_tokens=2000)
print(reply.text.strip())

# 정답은 보고서를 모두 열어 개요 표의 진행 상태를 세어야 나옵니다
ongoing = [doc_id for doc_id in CATALOG
           if doc_id.startswith('QIR-2026') and '| 진행 상태 | 진행 중 |' in read_doc(doc_id)]
print(f'\n정답: {len(ongoing)}건 {ongoing}')

검색된 청크의 문서: ['EDU-008', 'MTG-20260105', 'MTG-20261116', 'EDU-008', 'MTG-20260406']
**확인할 수 없습니다.** 제공된 문서에는 2026년 QIR의 진행 상태가 "진행 중"인지 명시된 건이 없습니다.

- **[MTG-20261116] 11월 3주차 주간 품질 회의**에서 진행 중인 QIR 점검이 있었지만, 각 QIR의 진행 상태 표기는 없습니다. 이 문서에서 언급된 건은 다음과 같습니다.
  - QIR-2026-018(BG256 기포): M-03 진공 펌프 교체 후 기포 불량 0.2%
  - QIR-2026-019(QX48 와이어 스윕 재발): 트랜스퍼 시간 14초 평가 후 유닛 비율 0.4%
  - QIR-2026-020(SP8 두께 미달): 보정 절차 누락 대응 방안 검토 중
  - QIR-2026-022(SP8 리드 도금 변색): 협력사 C 회신 관련 내용이 있으나 문서가 중간에 끊겨 있음
- 문서가 잘려 있어 이 회의에 더 많은 QIR이 있었는지 확인할 수 없습니다.
- **[EDU-008] 품질 이슈 보고서 작성 안내**에 따르면, 2주 안에 원인을 찾지 못한 QIR은 "진행 중"으로 두고 주간 품질 회의에서 경과를 보고합니다. 다만 위 문서만으로는 어느 QIR이 이 기준에 해당하는지 판단할 수 없습니다.

정확한 건수를 알려면 QIR 목록이나 각 보고서의 개요 항목(진행 상태)을 확인해야 합니다.

정답: 5건 ['QIR-2026-018', 'QIR-2026-019', 'QIR-2026-020', 'QIR-2026-021', 'QIR-2026-022']


## 예제 10-2. 위키 페이지 만들기

교재의 절: "LLM Wiki"

"QX48 와이어 스윕"을 다룬 원본 4건(보고서 두 건, 공지, 작업 표준)으로 위키 페이지 하나를 빌드합니다. 시스템 프롬프트에는 준비 코드의 위키 규칙(`WIKI_RULES`)을 넣습니다. 경과가 시간 순서로 모였는지, 문장마다 출처가 붙었는지, 원본끼리 값이 다른 곳에 [검토 필요]가 붙었는지 봅니다. 이 셀은 API를 1번 호출합니다.

In [ ]:
print(WIKI_RULES, end='\n\n')
sources = ['QIR-2025-031', 'QIR-2026-019', 'NOTI-2026-017', 'SOP-MD-001']   # "QX48 와이어 스윕"을 다룬 원본 4건
material = '\n\n'.join(f'<원본 id="{doc_id}">\n{read_doc(doc_id)}\n</원본>' for doc_id in sources)
page = llm.ask(f'{material}\n\n위 원본으로 "QX48 와이어 스윕" 위키 페이지를 작성해 주세요.',
               system=WIKI_RULES, model='소형', max_tokens=8000).text
print(page)

# 점검: 제목 줄을 뺀 줄 가운데 출처 문서 번호가 없는 줄을 셉니다
lines = [line for line in page.splitlines() if line.strip() and not line.startswith('#')]
unsourced = [line for line in lines if not re.search(r'\[(QIR|SOP|NOTI|MTG|MAN)-', line)]
print(f'\n제목을 뺀 줄 {len(lines)}개 가운데 출처가 없는 줄 {len(unsourced)}개, [검토 필요] {page.count("[검토 필요]")}곳')

너는 사내 문서로 위키를 만들고 고치는 편집자다. 다음 규칙을 지킨다.
1. 원본에 있는 내용만 쓴다. 원본에서 확인할 수 없는 내용은 쓰지 않는다.
2. 사실을 적은 문장마다 끝에 출처 문서 번호를 [QIR-2025-031]처럼 붙인다.
3. 원본끼리 값이나 기준이 다르면 한쪽으로 정하지 않는다. 그 문장 앞에 [검토 필요]를 붙이고 양쪽 값과 출처를 모두 적는다.
4. 페이지는 Markdown으로 쓰고 "## 요약", "## 경과", "## 현재 기준", "## 관련 문서" 절을 둔다.
5. 다른 위키 페이지가 될 만한 낱말은 [[페이지 이름]]으로 잇는다.

# QX48 와이어 스윕

## 요약

QX48은 몰딩 후 X선 검사에서 와이어 스윕 비율이 기준(5% 이하)을 넘는 유닛이 나온 제품이다. 2025년 6월에는 몰딩 프레스 M-02의 트랜스퍼 시간과 리드 프레임 변경이 원인으로 확인되어 트랜스퍼 시간을 10초에서 12초로 늘렸고, 해당 건은 2025-07-14에 종결되었다 [QIR-2025-031][SOP-MD-001]. 2026년 9월에는 9월 1일부터 바뀐 몰딩 컴파운드(CMP-B)와 관련된 것으로 보이는 와이어 스윕이 M-01과 M-02 모두에서 나왔으며, 원인은 아직 확인 중이다 [QIR-2026-019][NOTI-2026-017]. 트랜스퍼 시간 14초 평가가 진행 중이며, 평가가 끝날 때까지 QX48은 로트마다 X선 검사를 3스트립으로 늘려 운용한다 [QIR-2026-019].

## 경과

**2025년 6월: 1차 발생 (QIR-2025-031)**
- 2025-06-23 발행. 로트 L250619-02, L250620-01에서 와이어 스윕 비율 5%를 넘는 유닛이 4.2% 나왔다. 평소에는 0.5% 아래였다 [QIR-2025-031].
- 휜 와이어는 대부분 패키지의 게이트 반대쪽 모서리에 있었다 [QIR-2025-031].
- 원인: 몰딩 프레스 [[몰딩 프레스 M-02]]의 트랜스퍼 시간이 10초로, 같은 조건을 쓰는 다른 프레스

## 예제 10-3. 온톨로지로 트리플 뽑기

교재의 절: "온톨로지와 지식 그래프"

품질 이슈 보고서 18건에서 준비 코드의 온톨로지(`ONTOLOGY`)에 맞는 트리플을 뽑습니다. 관계별로 몇 개가 나왔는지, 사람이 원본을 읽고 정한 정답 트리플(`ANSWER_TRIPLES`)과 몇 개가 같은지 봅니다. API 오류가 났거나, 정상 종료하지 않았거나(잘림, 거절), 응답을 파싱하지 못한 보고서는 사유와 함께 `failed`에 남기고, 18건 가운데 몇 건에서 뽑았는지 출력합니다. `json.JSONDecodeError`는 `ValueError`의 하위 클래스이므로 `ValueError`로 함께 잡힙니다. 이 셀은 API를 18번(재시도하면 더) 호출합니다.

In [ ]:
print(ONTOLOGY, end='\n\n')
reports = [doc_id for doc_id in CATALOG if doc_id.startswith('QIR')]         # 품질 이슈 보고서 18건
failed = []                                                                 # 뽑지 못한 보고서와 사유


def extract(doc_id):
    """보고서 하나에서 온톨로지에 맞는 트리플을 뽑아 (주어, 관계, 대상)의 목록으로 돌려줍니다. 실패하면 failed에 적고 빈 목록.
    예외를 여기서 잡지 않으면 pool.map의 결과를 꺼낼 때 다시 발생해 받은 결과까지 잃습니다(5장)"""
    prompt = f'{ONTOLOGY}\n\n<원본 id="{doc_id}">\n{read_doc(doc_id)}\n</원본>\n\n이 원본에서 트리플을 모두 뽑아 주세요.'
    try:
        reply = llm.ask(prompt, model='소형', max_tokens=4000, schema=TRIPLE_SCHEMA)   # 관계 이름은 enum으로 제한
        if reply.stop_reason != 'end':          # 잘렸거나 거절한 응답은 파싱하지 않습니다(6장)
            raise ValueError(f'종료 이유 {reply.stop_reason}')
        return [(item['주어'], item['관계'], item['대상']) for item in json.loads(reply.text)['triples']]
    except (llm.CallError, ValueError, KeyError) as error:   # API 오류, 정상 종료가 아님, 파싱 오류, 필드 누락
        failed.append((doc_id, str(error)))
        return []


with ThreadPoolExecutor(max_workers=4) as pool:                             # 한 번에 4건까지 병렬로 보냅니다(5장)
    triples = [triple for found in pool.map(extract, reports) for triple in found]
print(f'뽑은 보고서 {len(reports) - len(failed)}/{len(reports)}건, 실패: {failed}')
print(f'트리플 {len(triples)}개, 관계별:', dict(Counter(relation for _, relation, _ in triples)))
print('QIR-2026-014의 트리플:', [triple for triple in triples if triple[0] == 'QIR-2026-014'])
print(f'정답 트리플 {len(ANSWER_TRIPLES)}개 가운데 같은 것 {len(ANSWER_TRIPLES & set(triples))}개')
print('정답에 없는 트리플:', sorted(set(triples) - ANSWER_TRIPLES))

## 예제 10-4. 관계를 따라 답하기

교재의 절: "온톨로지와 지식 그래프"

예제 10-3의 트리플에 문서 목록의 관리 부서를 더해 그래프를 만들고, 예제 10-1의 집계 질문, 제품별 보고서 수, 멀티홉 질문 하나에 답합니다. 멀티홉 질문은 "보고서가 두 건 이상 나온 설비에서, 재발 방지로 고친 문서와 그 문서의 관리 부서는?"입니다. API를 호출하지 않습니다.

In [ ]:
graph = triples + [(doc_id, '관리 부서', row['dept']) for doc_id, row in CATALOG.items()]   # 메타데이터는 LLM 없이 더함


def find(subject=None, relation=None, target=None):
    """조건에 맞는 트리플을 모두 돌려줍니다. None으로 둔 칸은 조건으로 쓰지 않습니다"""
    return [triple for triple in graph
            if subject in (None, triple[0]) and relation in (None, triple[1]) and target in (None, triple[2])]


ongoing = sorted(report for report, _, _ in find(relation='상태', target='진행 중'))
print(f'진행 중인 보고서 {len(ongoing)}건:', ongoing)
print('제품별 보고서 수:', dict(Counter(product for _, _, product in find(relation='제품'))))
counts = Counter(equipment for _, _, equipment in find(relation='발생 설비'))
for equipment in [name for name, count in counts.items() if count >= 2]:    # 보고서가 두 건 이상 나온 설비
    for report, _, _ in find(relation='발생 설비', target=equipment):
        for _, _, doc in find(report, '개정 문서'):
            print(f'{equipment} → {report} → {doc} → {[dept for _, _, dept in find(doc, "관리 부서")]}')

진행 중인 보고서 5건: ['QIR-2026-018', 'QIR-2026-019', 'QIR-2026-020', 'QIR-2026-021', 'QIR-2026-022']
제품별 보고서 수: {'QX48': 9, 'BG256': 5, 'SP8': 4}
M-02 → QIR-2025-031 → SOP-MD-001 → ['몰딩기술팀']
M-01 → QIR-2026-007 → SOP-MD-002 → ['몰딩기술팀']
A-01 → QIR-2026-016 → MAN-BA-001 → ['조립기술팀']
A-01 → QIR-2026-016 → SOP-BA-001 → ['조립기술팀']


## 실습 10-1. 새 문서로 위키 갱신하기

교재의 절: "개념 확인 실습"

예제 10-2의 페이지(`page`)에 2026-11-16 주간 품질 회의록(MTG-20261116)을 넣어 갱신하는 `update_page()`를 작성합니다. 이 회의록에는 QX48 트랜스퍼 시간을 11월 23일부터 14초로 바꾸기로 한 결정이 있고, 작업 표준 SOP-MD-001에는 아직 12초가 적혀 있습니다. 갱신한 페이지의 "현재 기준" 절에 트랜스퍼 시간이 어떻게 적히고 [검토 필요]가 붙을지 먼저 `my_guess`에 적습니다. 결과를 보고, 붙은 [검토 필요]를 사람이 어떻게 판정할지 한 문장으로 적으면 끝납니다.

In [ ]:
def update_page(page, doc_id):
    """위키 페이지와 새 원본 하나를 WIKI_RULES와 함께 보내, 고친 페이지 전체를 돌려받습니다"""
    return page  # TODO


my_guess = None  # TODO: "현재 기준"에 적힐 트랜스퍼 시간과 [검토 필요]가 붙을지를 먼저 적습니다
new_page = update_page(page, 'MTG-20261116')
current = new_page.split('## 현재 기준')[-1].split('\n## ')[0]      # "현재 기준" 절만 꺼냅니다
print(current.strip())
print(f'\n회의록 출처가 적힘: {"MTG-20261116" in current}, [검토 필요] {current.count("[검토 필요]")}곳 (예측 {my_guess})')

## 실습 10-2. 그래프로 멀티홉 질문에 답하기

교재의 절: "개념 확인 실습"

예제 10-4의 `find()`로 "제품 → 보고서 → 개정 문서 → 관리 부서"를 따라가는 `revised_docs()`를 작성합니다. 제품 하나를 받아 (보고서, 개정 문서, 관리 부서)의 목록을 돌려줍니다. BG256의 결과가 몇 행일지 먼저 `my_guess`에 적습니다. 결과를 보고, QX48의 결과에 와이어 스윕이 재발한 QIR-2026-019가 없는 이유를 설명하면 끝납니다.

In [ ]:
def revised_docs(product):
    """제품 → 보고서 → 개정 문서 → 관리 부서를 따라가 (보고서, 문서, 관리 부서)의 목록을 돌려줍니다"""
    return []  # TODO: find()로 세 관계를 차례로 따라갑니다


my_guess = None  # TODO: BG256의 결과가 몇 행일지 먼저 적습니다
for product in ['BG256', 'QX48']:
    rows = revised_docs(product)
    print(f'{product}: {len(rows)}행')
    for row in rows:
        print('  ', ' → '.join(row))
print(f'BG256 예측: {my_guess}행')